# Microsoft FCT Plots

This notebook handles both Microsoft HD and DM workloads with one shared plotting path.
Change workload-specific settings in `WORKLOAD_CONFIGS`; change experiment-wide settings in the shared constants.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from fct_parallel import run_fct_plot as run_single_fct_plot
from plot_utils import (
    DEFAULT_DPI,
    FIG_DIR,
    FONT_SIZE,
    TEXT_WIDTH,
    add_flow_size_regions,
    adjust_subplot_widths,
    metric_yticks,
    plot_variable_lines,
    save_and_trim,
    style_axis,
)

try:
    from IPython.display import display
except ImportError:
    display = print


In [ ]:
XTICKS_HD = [10**3, 10**4, 10**5, 10**6, 10**7, 10**8, 10**9]
XTICKS_DM = [10**2, 10**3, 10**4, 10**5, 10**6, 10**7, 10**8, 10**9]
YTICKS_HD_AVG = [10**-3, 10**-2, 10**-1, 10**0, 10**1, 10**2, 10**3]
YTICKS_HD_P99 = [10**-3, 10**-2, 10**-1, 10**0, 10**1, 10**2, 10**3, 10**4]
LONG_FIG_RATIO = 0.225
CUTOFF = 60_000_000
DEFAULT_ALPHA = 0.18
FCT_MAX_WORKERS = 3
VERBOSE = False

EXP_NAME = "FCT_microsoft"
NETWORKS = ["cbb_108i", "opera_ecmp", "clos_prio_128q"]
LABELS = ["CBB-Net", "Opera", "3:1 Fat-tree"]
VARIABLE_NAME = "load"
VARIABLE_VALUES = ["2.00", "4.00", "6.00", "8.00", "10.00"]
SEEDS = ["1", "2", "3", "4", "5"]
# SEEDS = ["4"]
METRICS = ["avg", "p99"]

WORKLOAD_CONFIGS = {
    "HD": {
        "file_template": "../../results/FCT_microsoft/log_{network}_HD_{variable}pload_seed={seed}.txt",
        "xticks": XTICKS_HD,
        "xlim": (XTICKS_HD[0], XTICKS_HD[-1]),
        "fig_suffix": "HD",
    },
    "DM": {
        "file_template": "../../results/FCT_microsoft/log_{network}_DM_{variable}pload_seed={seed}.txt",
        "xticks": XTICKS_DM,
        "xlim": (XTICKS_DM[0], 2 * 10**9),
        "fig_suffix": "DM",
    },
}


In [ ]:
# Common plotting helpers are imported from plot_utils.py.


In [ ]:
def plot_fct_results(
    fct_results_df,
    variable_name,
    variable_values,
    labels,
    fct_metric,
    fig_name,
    xticks,
    xlim,
):
    """Plot Microsoft FCT results for one workload/metric and save the figure."""
    num_networks = len(fct_results_df)
    if len(labels) != num_networks:
        raise ValueError("labels must have one entry for each network result")

    fig_width = TEXT_WIDTH
    fig_height = LONG_FIG_RATIO * fig_width
    first_yticks = metric_yticks(fct_metric, YTICKS_HD_AVG, YTICKS_HD_P99)
    normalized_yticks = [0.8, 1, 1.2, 1.4, 1.6, 1.8, 2]
    legend_handles = []
    legend_labels = []

    fig, axes = plt.subplots(
        1,
        num_networks,
        figsize=(fig_width, fig_height),
        sharey=False,
        dpi=DEFAULT_DPI,
    )
    axes = [axes] if num_networks == 1 else list(axes)

    for idx, (_, df) in enumerate(fct_results_df.items()):
        ax = axes[idx]
        add_flow_size_regions(ax, xlim, legend_handles, legend_labels)
        plot_variable_lines(ax, df, variable_name, variable_values, legend_handles, legend_labels)
        style_axis(ax, idx, labels[idx], fct_metric, xticks, xlim, first_yticks, normalized_yticks)

    fig.legend(
        handles=legend_handles,
        labels=legend_labels,
        handlelength=1.25,
        markerscale=1.0,
        loc="center right",
        fontsize=FONT_SIZE - 2,
        frameon=True,
    )
    plt.tight_layout(rect=[0, 0, 0.92, 1])
    adjust_subplot_widths(axes)

    save_and_trim(f"{FIG_DIR}/{fig_name}.png", dpi=DEFAULT_DPI)
    plt.show()


In [ ]:
def run_fct_plot(workload, fct_metric, display_network="opera_ecmp"):
    """Compute averaged FCT data and plot one Microsoft workload/metric pair."""
    workload = workload.upper()
    if workload not in WORKLOAD_CONFIGS:
        raise ValueError(f"Unknown workload {workload}; choose from {list(WORKLOAD_CONFIGS)}")

    config = WORKLOAD_CONFIGS[workload]
    return run_single_fct_plot(
        fct_metric,
        exp_name=f"{EXP_NAME}_{config['fig_suffix']}",
        file_template=config["file_template"],
        networks=NETWORKS,
        variable_name=VARIABLE_NAME,
        variable_values=VARIABLE_VALUES,
        seeds=SEEDS,
        labels=LABELS,
        xticks=config["xticks"],
        xlim=config["xlim"],
        plot_fct_results_func=plot_fct_results,
        display_func=display,
        display_network=display_network,
        max_workers=FCT_MAX_WORKERS,
        verbose=VERBOSE,
    )


def run_workload(workload):
    """Run both average and p99 plots for one workload."""
    return {metric: run_fct_plot(workload, metric) for metric in METRICS}


def run_all_workloads():
    """Run average and p99 plots for every configured Microsoft workload."""
    return {workload: run_workload(workload) for workload in WORKLOAD_CONFIGS}


## Run Selected Plots

Use the next cells to generate one workload at a time, or call `run_all_workloads()` to regenerate every Microsoft FCT figure.


In [ ]:
hd_results = run_workload("HD")


In [ ]:
dm_results = run_workload("DM")
